In [ ]:
import importlib, sys

def reimport_trackers():
    """Force-reload all trackers submodules already in sys.modules."""
    mods = [k for k in sys.modules if k.startswith("trackers")]
    for m in sorted(mods, reverse=True):
        importlib.reload(sys.modules[m])
    print(f"Reloaded {len(mods)} trackers module(s).")


# Phase 2 — Tracker ReID evaluation on MOT17 val

Measures the HOTA / MOTA / IDF1 / IDSW impact of adding the OSNet appearance encoder
to BoT-SORT and ByteTrack on the standard MOT17 half-val split.

**Configurations compared:**

| Config | Tracker | ReID | Fusion |
|---|---|---|---|
| BoT-SORT (baseline) | BoT-SORT | ✗ | — |
| BoT-SORT + ReID | BoT-SORT | ✓ | paper gated-min (§3.3) |
| ByteTrack (baseline) | ByteTrack | ✗ | — |
| ByteTrack + ReID | ByteTrack | ✓ | weighted (`appearance_weight=0.2`) |

CMC is disabled on BoT-SORT runs here to isolate the appearance contribution (the paper's
Table 1 ablation keeps CMC enabled).

**Caching:** tracker outputs and eval JSON are saved under `/content/trackers_reid_outputs`
and mirrored to `MyDrive/trackers_reid_outputs`. After a runtime restart, set
`RERUN["botsort_baseline"] = False` (etc.) to reload cached preds from Drive without re-tracking.

**Before running:** use a T4 GPU runtime, mount Drive with TrackEval MOT17 val GT + `img1/`
frames, and install a `trackers[reid]` build that includes gated-min BoT-SORT fusion and
ByteTrack ReID (see §1). Section §11 compares your runs against the paper's reported ReID gains.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before running.



## 1. Install

In [ ]:
!pip install -q --upgrade pip
# Needs gated-min BoT-SORT fusion + ByteTrack ReID (feat/reid-phase1 or newer).
!pip install -q --no-cache-dir 'trackers[reid] @ git+https://github.com/roboflow/trackers.git@feat/reid-phase1'

reimport_trackers()



In [ ]:
import warnings
import cv2
import numpy as np
import supervision as sv
import torch
from pathlib import Path

warnings.filterwarnings("ignore")
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")


## 2. Load ReID model

In [ ]:
from trackers.core.reid import ReIDModel

reid_model = ReIDModel.from_pretrained()   # OSNet x1.0 pretrained on MSMT17
print("ReID model loaded.")


## 3. Dataset — MOT17 val split

Same layout as `trackers metrics/mot17/trackers_all_tuning_both_state_estimators.ipynb`:

| Source | Location |
|---|---|
| **GT (val-half, frames 1–N)** | Drive `TrackEval/data/gt/MOT17_val_half/train_val` (or `gt_val_half.txt` under `MOT17/train_val`) |
| **Seqmap** | `TrackEval/data/gt/MOT17/MOT17-val.txt` |
| **Images (val-half)** | Drive mount — `MOT17-XX-FRCNN/img1/` |
| **YOLOX dets** | `gdown` zip → `/content/MOT17_yolox_dets/val/MOT17-XX_val.txt` |

YOLOX val dets use absolute frame IDs (302+); we remap them to 1..N to match val-half GT/images.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")


def discover_trackeval_mot17(root: Path) -> dict[str, Path]:
    """Find TrackEval MOT17 GT roots and val seqmap on Drive."""
    found: dict[str, Path] = {}
    if not root.exists():
        return found

    for seqmap in root.rglob("MOT17-val.txt"):
        if seqmap.parent.name != "MOT17":
            continue
        if "TrackEval" not in seqmap.parts:
            continue
        gt_mot17 = seqmap.parent
        gt_root = gt_mot17.parent
        found["seqmap"] = seqmap
        found["mot17_train_val"] = gt_mot17 / "train_val"
        val_half = gt_root / "MOT17_val_half" / "train_val"
        if val_half.exists():
            found["gt_trainval"] = val_half
        break

    return found


def discover_img_dirs(root: Path) -> dict[str, Path]:
    found: dict[str, Path] = {}
    if not root.exists():
        return found
    for img in root.rglob("img1"):
        seq = img.parent.name
        if seq.startswith("MOT17-") and any(img.glob("*.jpg")):
            found[seq] = img
    return found


def read_val_sequences(seqmap_path: Path) -> list[str]:
    lines = seqmap_path.read_text().strip().splitlines()
    return [ln.strip() for ln in lines[1:] if ln.strip()]


trackeval = discover_trackeval_mot17(MYDRIVE)
img_dirs = discover_img_dirs(MYDRIVE)

print("TrackEval on Drive:")
for k, p in trackeval.items():
    print(f"  {k}: {p}  (exists={p.exists()})")
print(f"Images: {len(img_dirs)} sequences")
if img_dirs:
    print("  example:", next(iter(img_dirs.items())))

if "seqmap" in trackeval:
    VAL_SEQUENCES = read_val_sequences(trackeval["seqmap"])
    print(f"Val seqmap ({len(VAL_SEQUENCES)}): {VAL_SEQUENCES}")
else:
    VAL_SEQUENCES = [
        "MOT17-02-FRCNN", "MOT17-04-FRCNN", "MOT17-05-FRCNN",
        "MOT17-09-FRCNN", "MOT17-10-FRCNN", "MOT17-11-FRCNN", "MOT17-13-FRCNN",
    ]
    print("WARNING: MOT17-val.txt not found — using default val list")



In [ ]:
!pip install -q gdown
import zipfile
import gdown

YOLOX_DIR = Path("/content/MOT17_yolox_dets")
YOLOX_VAL_DIR = YOLOX_DIR / "val"
ZIP_PATH = YOLOX_DIR / "yolox_bytetrack_detections_MOT17.zip"
FILE_ID = "1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT"

if not YOLOX_VAL_DIR.exists() or not any(YOLOX_VAL_DIR.glob("MOT17-*_val.txt")):
    YOLOX_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading YOLOX detections…")
    gdown.download(id=FILE_ID, output=str(ZIP_PATH), quiet=False)
    print("Extracting…")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(YOLOX_DIR)
    print("Done.")
else:
    print(f"YOLOX dets already present: {YOLOX_VAL_DIR}")

# MOT17-02_val.txt → MOT17-02-FRCNN
det_files: dict[str, Path] = {}
for det in sorted(YOLOX_VAL_DIR.glob("MOT17-*_val.txt")):
    base = det.stem.replace("_val", "")
    det_files[f"{base}-FRCNN"] = det

print(f"YOLOX val det files: {len(det_files)}")
print("  example:", next(iter(det_files.items())) if det_files else "none")



In [ ]:
import shutil

MOT17_ROOT = Path("/content/MOT17_val")
MOT17_ROOT.mkdir(parents=True, exist_ok=True)

MOT17_VAL_SEQMAP = trackeval.get("seqmap")
GT_TRAINVAL = trackeval.get("gt_trainval")
MOT17_TRAIN_VAL = trackeval.get("mot17_train_val")


def resolve_gt_source(seq: str) -> Path | None:
    """Prefer pre-built MOT17_val_half gt.txt; else gt_val_half.txt from MOT17/train_val."""
    if GT_TRAINVAL is not None:
        gt = GT_TRAINVAL / seq / "gt" / "gt.txt"
        if gt.exists():
            return gt
    if MOT17_TRAIN_VAL is not None:
        gt_half = MOT17_TRAIN_VAL / seq / "gt" / "gt_val_half.txt"
        if gt_half.exists():
            return gt_half
        gt = MOT17_TRAIN_VAL / seq / "gt" / "gt.txt"
        if gt.exists():
            return gt
    return None


def yolox_val_frame_offset(det_path: Path) -> int:
    """YOLOX val dets use absolute frame ids (302+); val-half GT/images use 1..N."""
    min_frame = None
    with open(det_path) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0]))
            min_frame = frame if min_frame is None else min(min_frame, frame)
    return (min_frame - 1) if min_frame and min_frame > 1 else 0


SEQUENCE_PATHS: dict[str, dict] = {}

for seq in VAL_SEQUENCES:
    src_gt = resolve_gt_source(seq)
    src_img = img_dirs.get(seq)
    det_path = det_files.get(seq)

    if src_gt is None:
        print(f"WARNING: no GT for {seq}")
        continue
    if src_img is None:
        print(f"WARNING: no img1 for {seq}")
        continue
    if det_path is None:
        print(f"WARNING: no YOLOX det for {seq}")
        continue

    seq_dir = MOT17_ROOT / seq
    gt_dir = seq_dir / "gt"
    img_dst = seq_dir / "img1"
    gt_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src_gt, gt_dir / "gt.txt")

    for base in (GT_TRAINVAL, MOT17_TRAIN_VAL):
        if base is None:
            continue
        seqinfo = base / seq / "seqinfo.ini"
        if seqinfo.exists():
            shutil.copy2(seqinfo, seq_dir / "seqinfo.ini")
            break

    if img_dst.exists() or img_dst.is_symlink():
        img_dst.unlink() if img_dst.is_symlink() else shutil.rmtree(img_dst)
    img_dst.symlink_to(src_img, target_is_directory=True)

    offset = yolox_val_frame_offset(det_path)
    n_imgs = len(list(src_img.glob("*.jpg")))

    SEQUENCE_PATHS[seq] = {
        "det": det_path,
        "img": img_dst,
        "gt": gt_dir / "gt.txt",
        "frame_offset": offset,
        "n_frames": n_imgs,
    }
    print(f"  {seq}: gt={src_gt.name}, {n_imgs} frames, det_offset={offset}")

ACTIVE_SEQUENCES = [
    s for s in VAL_SEQUENCES
    if s in SEQUENCE_PATHS
    and (MOT17_ROOT / s / "gt" / "gt.txt").exists()
]

print(f"\nReady to run ({len(ACTIVE_SEQUENCES)}/{len(VAL_SEQUENCES)}): {ACTIVE_SEQUENCES}")
if not ACTIVE_SEQUENCES:
    raise RuntimeError(
        "No sequences staged. Need TrackEval GT on Drive "
        "(TrackEval/data/gt/MOT17/…), val images (img1/), and YOLOX dets zip."
    )

OUTPUT_ROOT = Path("/content/trackers_reid_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ROOT = MYDRIVE / "trackers_reid_outputs"
LEGACY_DRIVE_OUTPUT_ROOT = MYDRIVE / "botsort_reid_outputs"
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Local outputs : {OUTPUT_ROOT}")
print(f"Drive cache   : {DRIVE_OUTPUT_ROOT}")

if MOT17_VAL_SEQMAP is None:
    MOT17_VAL_SEQMAP = OUTPUT_ROOT / "MOT17-val.txt"
    MOT17_VAL_SEQMAP.write_text("name\n" + "\n".join(ACTIVE_SEQUENCES) + "\n")
    print(f"Wrote fallback seqmap: {MOT17_VAL_SEQMAP}")
else:
    print(f"Using seqmap: {MOT17_VAL_SEQMAP}")




## 4. Tracking helpers


In [ ]:
import json
import shutil

from trackers import BoTSORTTracker, ByteTrackTracker
from trackers.eval import evaluate_mot_sequences
from trackers.eval.box import box_iou
from trackers.eval.results import BenchmarkResult
from trackers.io.mot import _MOTOutput, load_mot_file


# Set False to reload cached tracker outputs (restored from Drive when needed).
RERUN = {
    "botsort_baseline": False,
    "botsort_reid": True,
    "bytetrack_baseline": True,
    "bytetrack_reid": True,
}


def run_dir(name: str) -> Path:
    return OUTPUT_ROOT / name


def pred_dir_for(name: str) -> Path:
    return run_dir(name) / "preds"


def eval_cache_path(name: str) -> Path:
    return run_dir(name) / "eval_results.json"


def drive_run_dir(name: str) -> Path:
    return DRIVE_OUTPUT_ROOT / name


def legacy_drive_run_dir(name: str) -> Path:
    return LEGACY_DRIVE_OUTPUT_ROOT / name


def preds_complete(pred_dir: Path, sequences: list[str]) -> bool:
    return pred_dir.exists() and all((pred_dir / f"{s}.txt").exists() for s in sequences)


LEGACY_RUN_DIRS = {
    "botsort_baseline": ["baseline"],
    "botsort_reid": ["botsort_reid_w02"],
}


def restore_from_drive(name: str, sequences: list[str]) -> bool:
    """Copy preds + eval JSON from Drive into /content if local cache is missing."""
    local_preds = pred_dir_for(name)
    restored = False

    candidate_roots = [drive_run_dir(name)]
    for legacy in LEGACY_RUN_DIRS.get(name, []):
        candidate_roots.append(legacy_drive_run_dir(legacy))
        candidate_roots.append(drive_run_dir(legacy))

    for root in candidate_roots:
        drive_preds = root / "preds"
        if not preds_complete(local_preds, sequences) and preds_complete(drive_preds, sequences):
            print(f"  restoring {name} preds from Drive -> {local_preds}  (source={root})")
            local_preds.parent.mkdir(parents=True, exist_ok=True)
            if local_preds.exists():
                shutil.rmtree(local_preds)
            shutil.copytree(drive_preds, local_preds)
            restored = True
            break

    local_eval = eval_cache_path(name)
    for root in candidate_roots:
        drive_eval = root / "eval_results.json"
        if not local_eval.exists() and drive_eval.exists():
            local_eval.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(drive_eval, local_eval)
            restored = True
            break

    return restored


def publish_to_drive(name: str) -> None:
    """Mirror local preds + eval JSON to Drive for persistence across runtime restarts."""
    local_run = run_dir(name)
    drive_run = drive_run_dir(name)
    drive_run.mkdir(parents=True, exist_ok=True)

    local_preds = local_run / "preds"
    drive_preds = drive_run / "preds"
    if local_preds.exists():
        if drive_preds.exists():
            shutil.rmtree(drive_preds)
        shutil.copytree(local_preds, drive_preds)

    local_eval = local_run / "eval_results.json"
    if local_eval.exists():
        shutil.copy2(local_eval, drive_run / "eval_results.json")


def save_eval_result(name: str, result: BenchmarkResult) -> None:
    path = eval_cache_path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    result.save(path)


def load_eval_result(name: str) -> BenchmarkResult | None:
    path = eval_cache_path(name)
    if path.exists():
        return BenchmarkResult.load(path)
    return None


def load_yolox_val_dets(det_path: Path, frame_offset: int = 0) -> dict[int, sv.Detections]:
    """Load YOLOX val detections: frame,x1,y1,x2,y2,score -> {frame: Detections}."""
    frame_map: dict[int, list] = {}
    with open(det_path) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0])) - frame_offset
            if frame < 1:
                continue
            x1, y1, x2, y2, score = map(float, parts[1:6])
            if score <= 0:
                continue
            frame_map.setdefault(frame, []).append([x1, y1, x2, y2, score])
    return {
        frame: sv.Detections(
            xyxy=np.array(boxes, dtype=np.float32)[:, :4],
            confidence=np.array(boxes, dtype=np.float32)[:, 4],
        )
        for frame, boxes in frame_map.items()
    }


def run_sequences(
    name: str,
    tracker_factory,
    sequences: list[str],
    mot17_root: Path,
    output_dir: Path,
    use_frames: bool = False,
) -> Path:
    if not sequences:
        raise RuntimeError("ACTIVE_SEQUENCES is empty — re-run dataset staging cells.")

    pred_dir = pred_dir_for(name)
    pred_dir.mkdir(parents=True, exist_ok=True)

    for seq_name in sequences:
        spec = SEQUENCE_PATHS[seq_name]
        det_path = spec["det"]
        img_dir = spec["img"]
        offset = spec["frame_offset"]
        n_frames = spec["n_frames"]
        pred_path = pred_dir / f"{seq_name}.txt"

        det_data = load_yolox_val_dets(det_path, frame_offset=offset)
        frame_paths = sorted(img_dir.glob("*.jpg")) if use_frames else []

        tracker = tracker_factory()
        with _MOTOutput(pred_path) as mot:
            for frame_idx in range(1, n_frames + 1):
                dets = det_data.get(frame_idx, sv.Detections.empty())
                frame = None
                if use_frames and frame_idx <= len(frame_paths):
                    frame = cv2.imread(str(frame_paths[frame_idx - 1]))
                tracked = tracker.update(dets, frame)
                if tracked.tracker_id is not None:
                    tracked = tracked[tracked.tracker_id != -1]
                mot.write(frame_idx, tracked)

        n_dets = sum(len(v) for v in det_data.values())
        print(f"  {seq_name}: {n_frames} frames, {n_dets} detections")

    publish_to_drive(name)
    return pred_dir


def evaluate_preds(name: str, pred_dir: Path) -> BenchmarkResult:
    result = evaluate_mot_sequences(
        gt_dir=MOT17_ROOT,
        tracker_dir=pred_dir,
        seqmap=MOT17_VAL_SEQMAP,
        metrics=["CLEAR", "HOTA", "Identity"],
    )
    save_eval_result(name, result)
    publish_to_drive(name)
    return result


def load_or_run(name: str, tracker_factory, use_frames: bool = False) -> tuple[Path, BenchmarkResult]:
    """Run tracking + eval, or restore cached preds/eval from local disk or Drive."""
    restore_from_drive(name, ACTIVE_SEQUENCES)
    pred_dir = pred_dir_for(name)
    should_run = RERUN.get(name, True) or not preds_complete(pred_dir, ACTIVE_SEQUENCES)

    if should_run:
        print(f"Running {name}…")
        pred_dir = run_sequences(
            name=name,
            tracker_factory=tracker_factory,
            sequences=ACTIVE_SEQUENCES,
            mot17_root=MOT17_ROOT,
            output_dir=OUTPUT_ROOT,
            use_frames=use_frames,
        )
    else:
        print(f"Using cached preds for {name}: {pred_dir}")
        drive_preds = drive_run_dir(name) / "preds"
        if not preds_complete(drive_preds, ACTIVE_SEQUENCES):
            publish_to_drive(name)
            print(f"Mirrored {name} to Drive cache")

    cached = load_eval_result(name)
    if cached is not None and not should_run:
        print(f"Using cached eval for {name}")
        return pred_dir, cached

    print(f"Evaluating {name}…")
    return pred_dir, evaluate_preds(name, pred_dir)


def match_dets_to_gt_ids(
    gt_frame,
    det_xyxy: np.ndarray,
    min_iou: float = 0.5,
) -> np.ndarray:
    """Return GT track id per detection (-1 if unmatched / ignore)."""
    if len(det_xyxy) == 0:
        return np.array([], dtype=np.int64)

    gt_xyxy = sv.xywh_to_xyxy(gt_frame.boxes)
    keep = (gt_frame.confidences > 0) & (gt_frame.classes == 1)
    gt_xyxy = gt_xyxy[keep]
    gt_ids = gt_frame.ids[keep]

    if len(gt_xyxy) == 0:
        return np.full(len(det_xyxy), -1, dtype=np.int64)

    ious = box_iou(det_xyxy.astype(np.float64), gt_xyxy.astype(np.float64))
    assigned = np.full(len(det_xyxy), -1, dtype=np.int64)
    for det_i in range(len(det_xyxy)):
        best_j = int(np.argmax(ious[det_i]))
        if ious[det_i, best_j] >= min_iou:
            assigned[det_i] = int(gt_ids[best_j])
    return assigned


print("Helpers ready.")



## 5. BoT-SORT baseline (geometry only)



In [ ]:
def make_botsort_baseline():
    return BoTSORTTracker(enable_cmc=False, reid_model=None)

pred_dir_baseline, result_baseline = load_or_run(
    name="botsort_baseline",
    tracker_factory=make_botsort_baseline,
    use_frames=False,
)
print("Baseline ready.")



## 6. BoT-SORT + ReID  (paper gated-min fusion)



In [ ]:
def make_botsort_reid():
    return BoTSORTTracker(
        enable_cmc=False,
        reid_model=reid_model,
        reid_ema_alpha=0.9,
    )

pred_dir_reid, result_reid = load_or_run(
    name="botsort_reid",
    tracker_factory=make_botsort_reid,
    use_frames=True,
)
print("BoT-SORT + ReID ready.")



## 7. ByteTrack baseline (geometry only)



In [ ]:
def make_bytetrack_baseline():
    return ByteTrackTracker(reid_model=None)

pred_dir_bt_baseline, result_bt_baseline = load_or_run(
    name="bytetrack_baseline",
    tracker_factory=make_bytetrack_baseline,
    use_frames=False,
)
print("ByteTrack baseline ready.")



## 8. ByteTrack + ReID  (weighted fusion, w=0.2)



In [ ]:
def make_bytetrack_reid():
    return ByteTrackTracker(
        reid_model=reid_model,
        appearance_weight=0.2,
        reid_ema_alpha=0.9,
    )

pred_dir_bt_reid, result_bt_reid = load_or_run(
    name="bytetrack_reid",
    tracker_factory=make_bytetrack_reid,
    use_frames=True,
)
print("ByteTrack + ReID ready.")



## 9. ReID crop & embedding visualization



In [ ]:
!pip install -q matplotlib scikit-learn

import matplotlib.pyplot as plt
from matplotlib import patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.decomposition import PCA

# --- knobs ---
VIZ_SEQ = "MOT17-02-FRCNN"
VIZ_FRAME_STRIDE = 5          # sample every N frames
VIZ_MAX_FRAMES = 40           # cap number of frames sampled
VIZ_MIN_DET_CONF = 0.5
VIZ_MIN_IOU = 0.5             # IoU threshold for GT id assignment
VIZ_MAX_POINTS = 300          # cap scatter points (most recent kept)
VIZ_MAX_CROPS = 24            # crops shown in mosaic

spec = SEQUENCE_PATHS[VIZ_SEQ]
gt_data = load_mot_file(spec["gt"])
det_data = load_yolox_val_dets(spec["det"], frame_offset=spec["frame_offset"])
img_paths = sorted(spec["img"].glob("*.jpg"))

crops_rgb: list[np.ndarray] = []
embeddings: list[np.ndarray] = []
gt_ids: list[int] = []
frame_ids: list[int] = []

frame_list = list(range(1, spec["n_frames"] + 1, VIZ_FRAME_STRIDE))[:VIZ_MAX_FRAMES]
print(f"{VIZ_SEQ}: sampling {len(frame_list)} frames")

for frame_idx in frame_list:
    dets = det_data.get(frame_idx)
    gt_frame = gt_data.get(frame_idx)
    if dets is None or len(dets) == 0 or gt_frame is None:
        continue

    conf_mask = dets.confidence >= VIZ_MIN_DET_CONF
    dets = dets[conf_mask]
    if len(dets) == 0:
        continue

    frame_bgr = cv2.imread(str(img_paths[frame_idx - 1]))
    if frame_bgr is None:
        continue

    gt_for_frame = match_dets_to_gt_ids(gt_frame, dets.xyxy, min_iou=VIZ_MIN_IOU)
    embs = reid_model.extract_features(dets, frame_bgr)

    for i in range(len(dets)):
        if gt_for_frame[i] < 0:
            continue
        crop_bgr = sv.crop_image(frame_bgr, dets.xyxy[i].astype(int))
        if crop_bgr.size == 0:
            continue
        crops_rgb.append(crop_bgr[:, :, ::-1])
        embeddings.append(embs[i])
        gt_ids.append(int(gt_for_frame[i]))
        frame_ids.append(frame_idx)

if not embeddings:
    raise RuntimeError("No matched GT crops found — try lowering VIZ_MIN_IOU or VIZ_MIN_DET_CONF")

emb_matrix = np.stack(embeddings)
gt_arr = np.array(gt_ids)

# Keep at most VIZ_MAX_POINTS (spread across ids)
if len(emb_matrix) > VIZ_MAX_POINTS:
    keep = np.linspace(0, len(emb_matrix) - 1, VIZ_MAX_POINTS, dtype=int)
    emb_matrix = emb_matrix[keep]
    gt_arr = gt_arr[keep]
    crops_rgb = [crops_rgb[i] for i in keep]

coords = PCA(n_components=2, random_state=0).fit_transform(emb_matrix)

unique_ids = np.unique(gt_arr)
cmap = plt.colormaps["tab20"].resampled(max(len(unique_ids), 1))
id_to_color = {gid: cmap(i % 20) for i, gid in enumerate(unique_ids)}

fig = plt.figure(figsize=(14, 6))
gs = GridSpec(1, 2, width_ratios=[1.2, 1.0], wspace=0.25)

# --- PCA scatter ---
ax_scatter = fig.add_subplot(gs[0, 0])
for gid in unique_ids:
    mask = gt_arr == gid
    ax_scatter.scatter(
        coords[mask, 0], coords[mask, 1],
        s=28, alpha=0.85, color=id_to_color[gid], label=f"GT id {gid}",
    )
ax_scatter.set_title(f"{VIZ_SEQ} — PCA of ReID embeddings (colored by GT track id)")
ax_scatter.set_xlabel("PC1")
ax_scatter.set_ylabel("PC2")
ax_scatter.grid(True, alpha=0.3)
if len(unique_ids) <= 12:
    ax_scatter.legend(loc="best", fontsize=8, markerscale=1.2)

# --- crop mosaic ---
ax_crops = fig.add_subplot(gs[0, 1])
ax_crops.axis("off")
ax_crops.set_title(f"Sample crops ({min(len(crops_rgb), VIZ_MAX_CROPS)} shown)")

n_show = min(len(crops_rgb), VIZ_MAX_CROPS)
ncols = 6
nrows = int(np.ceil(n_show / ncols))
mosaic = np.ones((nrows * 64, ncols * 32, 3), dtype=np.uint8) * 255

for idx in range(n_show):
    r, c = divmod(idx, ncols)
    crop = cv2.resize(crops_rgb[idx], (32, 64))
    y0, x0 = r * 64, c * 32
    mosaic[y0:y0 + 64, x0:x0 + 32] = crop
    # color border by GT id
    color = (np.array(id_to_color[gt_arr[idx]])[:3] * 255).astype(np.uint8)
    mosaic[y0:y0 + 2, x0:x0 + 32] = color
    mosaic[y0 + 62:y0 + 64, x0:x0 + 32] = color
    mosaic[y0:y0 + 64, x0:x0 + 2] = color
    mosaic[y0:y0 + 64, x0 + 30:x0 + 32] = color

ax_crops.imshow(mosaic)
plt.tight_layout()
plt.show()

print(f"Points plotted: {len(coords)}  |  unique GT ids: {len(unique_ids)}  |  frames: {len(set(frame_ids))}")



## 10. Evaluate all configurations



In [ ]:
# Results were computed in load_or_run() above; reload from cache if needed.
RESULTS = {
    "botsort_baseline": "result_baseline",
    "botsort_reid": "result_reid",
    "bytetrack_baseline": "result_bt_baseline",
    "bytetrack_reid": "result_bt_reid",
}

for _name, _var in RESULTS.items():
    if globals().get(_var) is None:
        restore_from_drive(_name, ACTIVE_SEQUENCES)
        cached = load_eval_result(_name)
        if cached is None:
            raise RuntimeError(f"Missing eval cache for {_name}")
        globals()[_var] = cached

print("All eval results ready.")
for label, var in RESULTS.items():
    res = globals()[var]
    hota = res.aggregate.HOTA.HOTA * 100 if res.aggregate.HOTA else float("nan")
    print(f"  {label:<22} aggregate HOTA: {hota:.2f}%")



## 11. Comparison table



### Paper reference — BoT-SORT-ReID on MOT17 val (Table 1)

Aharon et al., *BoT-SORT: Robust Associations Multi-Pedestrian Tracking* ([arXiv:2206.14651](https://arxiv.org/abs/2206.14651)), Table 1 reports the **MOT17 validation split** (second half of train sequences) with YOLOX detections and the full BoT-SORT stack (updated KF + CMC + track prediction). Adding the ReID module gives **small but consistent identity gains**:

| Method | MOTA ↑ | IDF1 ↑ | HOTA ↑ |
|---|---:|---:|---:|
| BoT-SORT | 78.39 | 81.53 | 69.11 |
| BoT-SORT-ReID | 78.46 | 82.07 | 69.17 |
| **ReID Δ** | **+0.07** | **+0.54** | **+0.06** |

The paper describes ReID as a modest refinement on top of geometry/motion: detection accuracy (MOTA) moves slightly, while **IDF1** (identity preservation) improves the most. The ablation uses a **FastReID** encoder trained on MOT17 train-half crops; this notebook uses **OSNet (MSMT17)** and disables CMC on BoT-SORT, so absolute numbers will differ — compare **direction and magnitude** of the ReID delta, not the raw scores.

**ByteTrack + ReID** is **not** part of paper Table 1 (the paper's geometry baseline there is re-implemented ByteTrack at 77.66 MOTA / 79.77 IDF1 / 67.88 HOTA). We include it here as a second tracker using weighted appearance fusion.



In [ ]:
def fmt(r):
    a = r.aggregate
    hota  = a.HOTA.HOTA  * 100 if a.HOTA  else float("nan")
    assa  = a.HOTA.AssA  * 100 if a.HOTA  else float("nan")
    deta  = a.HOTA.DetA  * 100 if a.HOTA  else float("nan")
    mota  = a.CLEAR.MOTA * 100 if a.CLEAR else float("nan")
    idf1  = a.Identity.IDF1 * 100 if a.Identity else float("nan")
    idsw  = a.CLEAR.IDSW     if a.CLEAR else 0
    return hota, assa, deta, mota, idf1, idsw

configs = [
    ("BoT-SORT (baseline)", result_baseline),
    ("BoT-SORT + ReID",     result_reid),
    ("ByteTrack (baseline)", result_bt_baseline),
    ("ByteTrack + ReID",    result_bt_reid),
]

header = f"{'Config':<28}  {'HOTA':>6}  {'AssA':>6}  {'DetA':>6}  {'MOTA':>6}  {'IDF1':>6}  {'IDSW':>5}"
sep    = "-" * len(header)
print("Aggregate (all val sequences)")
print(header)
print(sep)
for label, res in configs:
    hota, assa, deta, mota, idf1, idsw = fmt(res)
    print(f"{label:<28}  {hota:6.2f}  {assa:6.2f}  {deta:6.2f}  {mota:6.2f}  {idf1:6.2f}  {idsw:5d}")
print(sep)

print("\nReID uplift in this notebook (ReID − baseline):")
pairs = [
    ("BoT-SORT", result_baseline, result_reid),
    ("ByteTrack", result_bt_baseline, result_bt_reid),
]
for name, base, reid in pairs:
    b = fmt(base)
    r = fmt(reid)
    print(
        f"  {name:<10}  ΔHOTA {r[0]-b[0]:+6.2f}  ΔMOTA {r[3]-b[3]:+6.2f}  "
        f"ΔIDF1 {r[4]-b[4]:+6.2f}  ΔIDSW {int(r[5]-b[5]):+5d}"
    )

print("\nPaper Table 1 reference (BoT-SORT-ReID, MOT17 val):  ΔHOTA +0.06  ΔMOTA +0.07  ΔIDF1 +0.54")



## 12. Per-sequence breakdown



In [ ]:
for seq in ACTIVE_SEQUENCES:
    def get(r, seq):
        s = r.sequences.get(seq)
        if s is None:
            return (float("nan"),) * 4
        hota = s.HOTA.HOTA  * 100 if s.HOTA  else float("nan")
        assa = s.HOTA.AssA  * 100 if s.HOTA  else float("nan")
        idf1 = s.Identity.IDF1 * 100 if s.Identity else float("nan")
        idsw = s.CLEAR.IDSW  if s.CLEAR else 0
        return hota, assa, idf1, idsw

    rows = [
        ("BoT-SORT (baseline)", get(result_baseline, seq)),
        ("BoT-SORT + ReID",     get(result_reid, seq)),
        ("ByteTrack (baseline)", get(result_bt_baseline, seq)),
        ("ByteTrack + ReID",    get(result_bt_reid, seq)),
    ]

    print(f"{seq}")
    print(f"  {'Config':<28}  {'HOTA':>6}  {'AssA':>6}  {'IDF1':>6}  {'IDSW':>5}")
    for label, (hota, assa, idf1, idsw) in rows:
        print(f"  {label:<28}  {hota:6.2f}  {assa:6.2f}  {idf1:6.2f}  {idsw:5d}")
    print()

